# Step 1 — Hinglish lecture ASR: MUCS 2021 Hi-En test → windows → ASR → WER / CER / transliterated WER

Paper: *How much do code-switched ASR errors damage summaries of Hinglish lecture audio?*

What this notebook does, in order:
1. Downloads the **MUCS 2021 Hindi-English test split** (OpenSLR SLR104, CC BY-SA 4.0, ~443 MB).
2. Parses the Kaldi files (`text`, `segments`, `wav.scp`), flags duplicate utterances/recordings.
3. Builds **5–10 min windows** of contiguous utterances per recording and selects `N_WINDOWS` (seeded).
   Audio = the exact transcribed utterance spans concatenated with a short silence, so the oracle transcript
   covers all of the audio (no untranscribed speech inflating insertions).
4. Runs **whisper-small** and one **Hinglish fine-tune** with faster-whisper (deterministic decoding).
5. Computes per-window **WER, CER, transliterated WER/CER**, per-language token error rates, corpus-level
   values with **bootstrap CIs**, and a **paired test** between the two models.

Everything is written to CSV/JSONL. Every ASR decoding prompt/config is appended to `logs/prompts_log.jsonl`.

**Runtime:** Runtime → Change runtime type → **T4 GPU**. Re-running is safe: finished work is skipped.

Nothing in this notebook reports a number it did not compute. Model-card claims quoted in comments are
marked as such.

In [8]:
# ===================== CONFIG (edit here only) =====================
SEED = 13
N_WINDOWS = 25              # target number of 5-10 min windows (paper scope: 20-30)
MIN_WIN_SEC = 300.0         # min speech per window
MAX_WIN_SEC = 600.0         # max speech per window
MIN_EN_FRAC = 0.05          # window must contain >=5% Latin-script (English) tokens (needed for EN noise sweep)
MIN_HI_FRAC = 0.30          # ...and >=30% Devanagari (Hindi) tokens (needed for HI noise sweep)
GAP_SEC = 0.2               # silence inserted between concatenated utterances
DUP_JACCARD = 0.5           # recordings sharing >=50% of normalised utterance texts are treated as duplicates
N_BOOT = 2000               # bootstrap resamples

USE_DRIVE = True            # outputs + logs to Google Drive so they survive a Colab reset
DRIVE_ROOT = "/content/drive/MyDrive/hinglish_asr_sum"
LOCAL_ROOT = "/content/hinglish_asr_sum"
DATA_ROOT = "/content/mucs_data"       # raw data + windows stay on local disk (fast, re-creatable)

# OpenSLR SLR104 mirrors (from https://openslr.org/104/)
MUCS_URLS = [
    "https://openslr.trmal.net/resources/104/Hindi-English_test.tar.gz",
    "https://openslr.elda.org/resources/104/Hindi-English_test.tar.gz",
    "https://openslr.magicdatatech.com/resources/104/Hindi-English_test.tar.gz",
]

# Common decoding settings: forced language, deterministic (no temperature fallback), no text prompt.
_BASE_DECODE = dict(task="transcribe", beam_size=5, temperature=0.0,
                    condition_on_previous_text=False, vad_filter=False, initial_prompt=None)

ASR_MODELS = {
    # Stock multilingual baseline (faster-whisper downloads Systran/faster-whisper-small).
    "whisper-small": dict(
        source="small", compute_type="float16",
        decode=dict(_BASE_DECODE, language="hi", without_timestamps=False)),
    # Hinglish fine-tune of whisper-large-v3 (Apache-2.0). Model card: outputs Hindi in Devanagari and English
    # in Latin (same convention as MUCS references). Training data NOT disclosed on the card -> contamination
    # with MUCS cannot be ruled out (see checklist). Card recommends an extra <|mixedcode|> token after <|hi|>;
    # faster-whisper cannot insert it, so we decode with <|hi|> only (a documented limitation).
    "trelis-hinglish-preview": dict(
        source="convert:Trelis/whisper-hinglish-preview", compute_type="float16",
        decode=dict(_BASE_DECODE, language="hi", without_timestamps=False)),
    # FALLBACK if the large model OOMs: whisper-base fine-tune (MIT) with CTranslate2 weights on the hub.
    # Card: ALWAYS outputs Roman script, decode with language="en", without_timestamps=True;
    # trained on MUCS 2021 Hi-En *train* (86 h) -> in-domain; only transliterated metrics are meaningful.
    "abhishek-whisper-base-hinglish": dict(
        source="hfct2:abhishekgautamm/whisper-base-hinglish:ct2/fp32", compute_type="float16",
        decode=dict(_BASE_DECODE, language="en", without_timestamps=True,
                    no_repeat_ngram_size=3, repetition_penalty=1.1)),
}
RUN_MODELS = ["whisper-small", "trelis-hinglish-preview"]   # baseline first, fine-tune second

# Long-form segmentation.
# "fixed30" (v1): faster-whisper's default 30 s windows. The fine-tune emits no timestamps, so it emitted exactly one
#   segment per 30 s window and anything after an early stop / token limit inside a window was skipped.
# "vad" (v2, default): Silero VAD (bundled with faster-whisper) finds speech; chunks are merged into clips of at most
#   MAX_CLIP_SEC, cut only at pauses; each clip is decoded on its own, without timestamps, for BOTH models.
SEGMENTATION = "vad"
MAX_CLIP_SEC = 15.0
VAD_PARAMS = dict(min_silence_duration_ms=150, speech_pad_ms=100, max_speech_duration_s=MAX_CLIP_SEC)
ASR_TAG = f"vad{int(MAX_CLIP_SEC)}" if SEGMENTATION == "vad" else "fixed30"   # suffix of all ASR output files

In [9]:
# ===================== INSTALL + ENVIRONMENT =====================
import subprocess, sys, importlib
PINS = ["faster-whisper==1.2.1", "jiwer==4.0.0", "indic_transliteration==2.3.82"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
importlib.invalidate_caches()

import os, re, io, gc, json, glob, time, zlib, random, shutil, hashlib, tarfile, platform, unicodedata
from datetime import datetime, timezone
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
print("GPU:", gpu or "NONE (ASR will be very slow on CPU — switch runtime to T4)")

# ctranslate2 (faster-whisper's engine) dlopens libcublas.so.12 at first GPU use. If the Colab image ships a
# different CUDA major version, that fails with "libcublas.so.12 is not found". Fix: install NVIDIA's CUDA 12 cuBLAS
# wheel and preload it by full path with RTLD_GLOBAL, so the later lookup by name finds the loaded copy.
import ctypes
def ensure_cublas12():
    try:
        ctypes.CDLL("libcublas.so.12"); return "system"
    except OSError:
        pass
    def find():
        return sorted({p for d in sys.path if d for p in glob.glob(f"{d}/nvidia/cublas/lib/libcublas.so.12")})
    if not find():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12==12.9.2.10"], check=True)
        importlib.invalidate_caches()
    hits = find()
    assert hits, "nvidia-cublas-cu12 installed but libcublas.so.12 not found on sys.path"
    libdir = os.path.dirname(hits[0])
    for name in ("libcublasLt.so.12", "libcublas.so.12"):          # dependency first
        ctypes.CDLL(f"{libdir}/{name}", mode=ctypes.RTLD_GLOBAL)
    return libdir
CUBLAS_SOURCE = ensure_cublas12() if gpu else "n/a (cpu)"
print("cuBLAS 12:", CUBLAS_SOURCE)

GPU: Tesla T4, 15360 MiB
cuBLAS 12: /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib


In [10]:
# ===================== PATHS, SEEDS, LOGGING =====================
IN_COLAB = "google.colab" in sys.modules
ROOT = LOCAL_ROOT
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = DRIVE_ROOT
OUT_DIR, LOG_DIR = f"{ROOT}/outputs", f"{ROOT}/logs"
RAW_DIR, WIN_DIR, MODELS_DIR = f"{DATA_ROOT}/raw", f"{DATA_ROOT}/windows", f"{DATA_ROOT}/models"
for d in (OUT_DIR, LOG_DIR, RAW_DIR, WIN_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)

random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass

PROMPT_LOG = f"{LOG_DIR}/prompts_log.jsonl"
RUN_CONFIG = f"{LOG_DIR}/run_config_step1.json"

def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def log_prompt(step, component, model, prompt, params):
    """Append one prompt/decoding record. Step 2 uses the same function for LLM prompts."""
    rec = dict(ts=now_utc(), step=step, component=component, model=model, prompt=prompt, params=params)
    with open(PROMPT_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def update_run_config(**kv):
    cfg = json.load(open(RUN_CONFIG)) if os.path.exists(RUN_CONFIG) else {}
    cfg.update(kv)
    json.dump(cfg, open(RUN_CONFIG, "w"), indent=2, ensure_ascii=False, default=str)

def pkg_version(name):
    try:
        import importlib.metadata as md; return md.version(name)
    except Exception:
        return None

update_run_config(
    started=now_utc(), python=platform.python_version(), gpu=gpu, seed=SEED, cublas12=CUBLAS_SOURCE,
    config=dict(N_WINDOWS=N_WINDOWS, MIN_WIN_SEC=MIN_WIN_SEC, MAX_WIN_SEC=MAX_WIN_SEC, MIN_EN_FRAC=MIN_EN_FRAC,
                MIN_HI_FRAC=MIN_HI_FRAC, GAP_SEC=GAP_SEC, DUP_JACCARD=DUP_JACCARD, N_BOOT=N_BOOT,
                RUN_MODELS=RUN_MODELS, ASR_MODELS=ASR_MODELS),
    packages={p: pkg_version(p) for p in ["faster-whisper", "ctranslate2", "jiwer", "indic_transliteration",
                                          "transformers", "numpy", "pandas", "scipy", "soundfile"]})
print("Outputs ->", OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs -> /content/drive/MyDrive/hinglish_asr_sum/outputs


In [11]:
# --- RESUME PRE-CHECK ---
import shutil
assert ASR_TAG == "vad15", ASR_TAG
shutil.copy(f"{OUT_DIR}/selected_windows.csv", f"{OUT_DIR}/selected_windows_before_resume.csv")
shutil.copy(f"{OUT_DIR}/asr_hyps_vad15.csv", f"{OUT_DIR}/asr_hyps_vad15_backup_46rows.csv")
_h = pd.read_csv(f"{OUT_DIR}/asr_hyps_vad15.csv", keep_default_na=False)
print(_h.groupby("model").window_id.nunique())
print("Trelis missing:", sorted({f"W{i:02d}" for i in range(25)}
      - set(_h[_h.model == "trelis-hinglish-preview"].window_id)))

model
trelis-hinglish-preview    21
whisper-small              25
Name: window_id, dtype: int64
Trelis missing: ['W21', 'W22', 'W23', 'W24']


In [12]:
# ===================== DOWNLOAD + EXTRACT MUCS Hi-En TEST =====================
TAR_PATH = f"{RAW_DIR}/Hindi-English_test.tar.gz"
EXTRACT_DIR = f"{RAW_DIR}/extracted"

def sha256_file(path, bufsize=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(bufsize), b""):
            h.update(chunk)
    return h.hexdigest()

if not os.path.exists(TAR_PATH):
    for url in MUCS_URLS:
        print("Downloading", url)
        r = subprocess.run(["wget", "-q", "--show-progress", "-O", TAR_PATH + ".part", url])
        if r.returncode == 0 and os.path.getsize(TAR_PATH + ".part") > 100_000_000:
            os.rename(TAR_PATH + ".part", TAR_PATH); break
        print("  failed, trying next mirror")
    else:
        raise RuntimeError("All mirrors failed. Download Hindi-English_test.tar.gz manually from "
                           "https://openslr.org/104/ and upload it to " + TAR_PATH)

if not os.path.exists(f"{EXTRACT_DIR}/.done"):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with tarfile.open(TAR_PATH) as tf:
        try:
            tf.extractall(EXTRACT_DIR, filter="data")   # safe extraction (Python >= 3.12)
        except TypeError:
            tf.extractall(EXTRACT_DIR)
    open(f"{EXTRACT_DIR}/.done", "w").close()

tar_sha = sha256_file(TAR_PATH)
update_run_config(mucs_tar=dict(path=TAR_PATH, bytes=os.path.getsize(TAR_PATH), sha256=tar_sha))
print("tar sha256:", tar_sha)

tar sha256: 93e358b3bf8233a897fcd353c1f4f98fdda6b8c01b7eed17a70c7dd26e984b37


In [13]:
# ===================== PARSE KALDI FILES =====================
def natural_key(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]

def find_named(root, name):
    return sorted(p for p in glob.glob(f"{root}/**/{name}", recursive=True) if os.path.isfile(p))

all_wavs = sorted(glob.glob(f"{EXTRACT_DIR}/**/*.wav", recursive=True))
wav_by_base = defaultdict(list)
for p in all_wavs:
    wav_by_base[os.path.basename(p)].append(p)

text_files = find_named(EXTRACT_DIR, "text")
print("Found text files:", text_files)
print("Found segments files:", find_named(EXTRACT_DIR, "segments"))
print("Found wav.scp files:", find_named(EXTRACT_DIR, "wav.scp"))
print("Found", len(all_wavs), "wav files")
assert text_files, "No Kaldi 'text' file found — inspect the extracted tree (printed above) and adapt this cell."
KALDI_DIR = os.path.dirname(text_files[0])
if len(text_files) > 1:
    print("WARNING: several 'text' files; using", text_files[0])

def read_kv(path, maxsplit=1):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(line.split(None, maxsplit))
    return rows

utt_text = {r[0]: (r[1] if len(r) > 1 else "") for r in read_kv(f"{KALDI_DIR}/text")}

def resolve_wav(entry):
    """wav.scp value -> existing path. Handles plain paths and simple command pipes."""
    cands = [t for t in entry.split() if t.lower().endswith(".wav")] or [entry]
    for c in cands:
        for p in (c, os.path.join(KALDI_DIR, c), os.path.join(EXTRACT_DIR, c)):
            if os.path.isfile(p):
                return p
        hits = wav_by_base.get(os.path.basename(c), [])
        if len(hits) == 1:
            return hits[0]
    return None

seg_path, scp_path = f"{KALDI_DIR}/segments", f"{KALDI_DIR}/wav.scp"
rows = []
if os.path.exists(seg_path):
    SEG_MODE = "kaldi_segments"
    reco_wav = {}
    if os.path.exists(scp_path):
        for r in read_kv(scp_path):
            reco_wav[r[0]] = resolve_wav(r[1] if len(r) > 1 else r[0])
    for utt, reco, st, en in (r for r in read_kv(seg_path, 3)):
        path = reco_wav.get(reco) or resolve_wav(reco + ".wav")
        rows.append(dict(utt_id=utt, reco_id=reco, wav_path=path, start=float(st), end=float(en)))
else:
    # Fallback: one wav per utterance; recording id = utterance id minus its last "_suffix". CHECK THIS.
    SEG_MODE = "utterance_wavs_fallback"
    import soundfile as sf
    for utt in utt_text:
        path = resolve_wav(utt + ".wav")
        if path:
            rows.append(dict(utt_id=utt, reco_id=utt.rsplit("_", 1)[0], wav_path=path,
                             start=0.0, end=sf.info(path).duration))
    print("WARNING: no segments file — using fallback grouping. Verify reco_id grouping manually!")

utts = pd.DataFrame(rows)
utts["text_raw"] = utts.utt_id.map(utt_text)
n0 = len(utts)
missing = utts.wav_path.isna() | utts.text_raw.isna()
utts = utts[~missing].copy()
utts["dur"] = utts.end - utts.start
utts["order"] = utts.utt_id.map(natural_key)
print(f"mode={SEG_MODE} | utterances={n0} (dropped {missing.sum()} without audio/text) | "
      f"recordings={utts.reco_id.nunique()} | speech={utts.dur.sum()/3600:.2f} h")
assert utts.dur.min() > 0, "non-positive utterance durations — check segments file"
update_run_config(kaldi=dict(dir=KALDI_DIR, mode=SEG_MODE, n_utts=len(utts), n_dropped=int(missing.sum()),
                             n_recordings=int(utts.reco_id.nunique()), speech_hours=float(utts.dur.sum() / 3600)))

Found text files: ['/content/mucs_data/raw/extracted/test/transcripts/text']
Found segments files: ['/content/mucs_data/raw/extracted/test/transcripts/segments']
Found wav.scp files: ['/content/mucs_data/raw/extracted/test/transcripts/wav.scp']
Found 30 wav files
mode=kaldi_segments | utterances=3136 (dropped 0 without audio/text) | recordings=30 | speech=5.18 h


In [14]:
# ===================== TEXT NORMALISATION, SCRIPT TAGGING, ROMANISATION =====================
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

DEVA_RE = re.compile(r"[ऀ-ॿ]")
LATIN_RE = re.compile(r"[a-z]")
DEVA_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
DEVA_CONSONANTS = set(chr(c) for c in list(range(0x0915, 0x093A)) + list(range(0x0958, 0x0960)))
NUKTA = "़"

def norm_text(s):
    """Script-safe normalisation used for ALL metrics: NFC, drop <tags>/[tags], lowercase, Devanagari digits ->
    ASCII, every char that is not a letter/mark/number becomes a space (removes danda, punctuation, ZWJ/ZWNJ).
    Keeps Devanagari matras and nuktas (unlike Whisper's default normaliser)."""
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"<[^>]*>|\[[^\]]*\]", " ", s).lower().translate(DEVA_DIGITS)
    s = "".join(ch if unicodedata.category(ch)[0] in "LMN" else " " for ch in s)
    return " ".join(s.split())

def token_script(tok):
    if DEVA_RE.search(tok):
        return "hi"
    if LATIN_RE.search(tok):
        return "en"
    return "other"      # digits etc. — language-independent tokens

def romanize_token(tok):
    """Lossy, deterministic Devanagari -> ASCII romanisation (IAST, word-final schwa deletion,
    diacritics stripped, nukta/anusvara/candrabindu merged). Latin tokens unchanged. Token count preserved."""
    if not DEVA_RE.search(tok):
        return tok
    t = tok.replace(NUKTA, "")
    r = transliterate(t, sanscript.DEVANAGARI, sanscript.IAST)
    if len(t) > 1 and t[-1] in DEVA_CONSONANTS and r.endswith("a"):
        r = r[:-1]                                    # word-final inherent schwa is not pronounced
    for a, b in (("ṃ", "n"), ("ṁ", "n"), ("~", "n"), ("ś", "sh"), ("ṣ", "sh"), ("ṛ", "ri"), ("ḥ", "h")):
        r = r.replace(a, b)
    r = re.sub(r"c(?!h)", "ch", r)
    r = "".join(ch for ch in unicodedata.normalize("NFD", r) if not unicodedata.combining(ch)).lower()
    r = re.sub(r"[^a-z0-9]", "", r)
    return r or tok

def romanize_text(normed):
    return " ".join(romanize_token(t) for t in normed.split())

def script_stats(normed):
    toks = normed.split()
    c = Counter(token_script(t) for t in toks)
    n, u = len(toks), c["other"]
    lang = n - u
    cmi = 0.0 if lang == 0 else 100.0 * (1 - max(c["hi"], c["en"]) / lang)   # Das & Gambäck CMI
    return dict(n_tokens=n, frac_hi=c["hi"] / n if n else 0.0, frac_en=c["en"] / n if n else 0.0,
                frac_other=u / n if n else 0.0, cmi=cmi)

# Self-tests (fail loudly if a library update changes behaviour)
assert norm_text("Hello, दुनिया। <noise> [laugh] ३ बार!") == "hello दुनिया 3 बार"
assert romanize_text("क्या है निष्पादित statement") == "kya hai nishpadit statement"
assert len(romanize_text("आज मैंने new laptop लिया").split()) == 5
print("romanisation examples:", {w: romanize_token(w) for w in
      ["सीन", "कंडीशन", "हूँ", "ज़रूरत", "मूल्यांकित", "फाइल", "न"]})

utts["text_norm"] = utts.text_raw.map(norm_text)
utts = utts[utts.text_norm.str.len() > 0].copy()

romanisation examples: {'सीन': 'sin', 'कंडीशन': 'kandishan', 'हूँ': 'hun', 'ज़रूरत': 'jarurat', 'मूल्यांकित': 'mulyankit', 'फाइल': 'phail', 'न': 'na'}


In [15]:
# ===================== DUPLICATES, WINDOWS, SELECTION =====================
# 1) utterance-level exact duplicates (reported, not removed: they are real audio)
dup_utt_frac = float(utts.text_norm.duplicated(keep=False).mean())
# 2) recording-level near-duplicates (same tutorial recorded twice): keep one per group
reco_sets = utts.groupby("reco_id").text_norm.apply(set).to_dict()
recos = sorted(reco_sets, key=natural_key)
parent = {r: r for r in recos}
def find(r):
    while parent[r] != r:
        parent[r] = parent[parent[r]]; r = parent[r]
    return r
dup_pairs = []
for i, a in enumerate(recos):
    for b in recos[i + 1:]:
        A, B = reco_sets[a], reco_sets[b]
        j = len(A & B) / len(A | B)
        if j >= DUP_JACCARD:
            dup_pairs.append((a, b, round(j, 3)))
            parent[find(b)] = find(a)
keep_recos = sorted({find(r) for r in recos}, key=natural_key)
print(f"exact-duplicate utterance texts: {dup_utt_frac:.1%} | near-duplicate recording pairs: {len(dup_pairs)} "
      f"| recordings kept: {len(keep_recos)}/{len(recos)}")
pd.DataFrame(dup_pairs, columns=["reco_a", "reco_b", "jaccard"]).to_csv(f"{OUT_DIR}/duplicate_recordings.csv",
                                                                         index=False)

# 3) windows: contiguous utterances in recording order, emitted once speech >= MIN_WIN_SEC
order_col = "start" if SEG_MODE == "kaldi_segments" else "order"
windows = []
for reco in keep_recos:
    g = utts[utts.reco_id == reco].sort_values(order_col)
    cur = []
    for row in g.itertuples():
        cur.append(row)
        speech = sum(r.dur for r in cur)
        if speech >= MIN_WIN_SEC:
            if speech <= MAX_WIN_SEC:
                windows.append(cur)
            cur = []
win_rows = []
for k, cur in enumerate(windows):
    ref_norm = " ".join(r.text_norm for r in cur)
    st = script_stats(ref_norm)
    win_rows.append(dict(
        reco_id=cur[0].reco_id, first_utt=cur[0].utt_id, last_utt=cur[-1].utt_id, n_utts=len(cur),
        speech_sec=round(sum(r.dur for r in cur), 3),
        source_span_sec=round(cur[-1].end - cur[0].start, 3) if SEG_MODE == "kaldi_segments" else np.nan,
        utt_ids=" ".join(r.utt_id for r in cur), ref_raw=" ".join(r.text_raw for r in cur), ref_norm=ref_norm,
        **st))
cands = pd.DataFrame(win_rows)
assert len(cands), "No windows built — lower MIN_WIN_SEC or check durations."
cands["eligible"] = (cands.frac_en >= MIN_EN_FRAC) & (cands.frac_hi >= MIN_HI_FRAC)
print(f"candidate windows: {len(cands)} | eligible: {int(cands.eligible.sum())} "
      f"(rejected for too little EN: {int((cands.frac_en < MIN_EN_FRAC).sum())}, "
      f"too little HI: {int((cands.frac_hi < MIN_HI_FRAC).sum())})")

# 4) seeded round-robin selection: at most one window per recording per round -> topic/speaker diversity
rng_sel = random.Random(SEED)
elig = cands[cands.eligible]
by_reco = {r: list(g.index) for r, g in elig.groupby("reco_id")}
reco_order = sorted(by_reco, key=natural_key); rng_sel.shuffle(reco_order)
chosen, rnd = [], 0
while len(chosen) < N_WINDOWS and any(len(v) > rnd for v in by_reco.values()):
    for r in reco_order:
        if rnd < len(by_reco[r]) and len(chosen) < N_WINDOWS:
            chosen.append(by_reco[r][rnd])
    rnd += 1
if len(chosen) < N_WINDOWS:
    print(f"WARNING: only {len(chosen)} eligible windows (< N_WINDOWS={N_WINDOWS}).")
sel = cands.loc[chosen].copy().reset_index(drop=True)
sel.insert(0, "window_id", [f"W{i:02d}" for i in range(len(sel))])
cands.to_csv(f"{OUT_DIR}/candidate_windows.csv", index=False)
sel.to_csv(f"{OUT_DIR}/selected_windows.csv", index=False)
print(sel[["window_id", "reco_id", "n_utts", "speech_sec", "frac_hi", "frac_en", "cmi"]].to_string(index=False))
print(f"total selected speech: {sel.speech_sec.sum()/60:.1f} min")
update_run_config(selection=dict(n_candidates=len(cands), n_eligible=int(cands.eligible.sum()),
                                 n_selected=len(sel), dup_utt_frac=dup_utt_frac, n_dup_reco_pairs=len(dup_pairs),
                                 selected_minutes=float(sel.speech_sec.sum() / 60)))

exact-duplicate utterance texts: 12.8% | near-duplicate recording pairs: 0 | recordings kept: 30/30
candidate windows: 48 | eligible: 42 (rejected for too little EN: 6, too little HI: 0)
window_id          reco_id  n_utts  speech_sec  frac_hi  frac_en       cmi
      W00 fpL6MBSZsci9dyAn      67       300.0 0.889561 0.107413 10.773900
      W01 406yMKxIdSDHRf8H      57       303.0 0.728244 0.267176 26.840491
      W02 dNHXkqqVPqWYhzis      52       301.0 0.694620 0.300633 30.206677
      W03 NhV290ew2hDJfVqc      58       301.0 0.667224 0.327759 32.941176
      W04 x8iijxWeCf26KNpC      53       302.0 0.673171 0.323577 32.463295
      W05 4xyIm2P6Xzlin341      58       302.0 0.907767 0.087379  8.780488
      W06 UO5ZLDtkc8QMi9Hk      47       301.0 0.634921 0.343254 35.091278
      W07 TkzR0GrQyK0vSTuj      41       317.0 0.865320 0.127946 12.881356
      W08 RRrEfkw4Rh5TdJkS      46       311.0 0.679598 0.316092 31.746032
      W09 w5Jyq3XMbb3WwiKQ      55       303.0 0.745626 0.24899

In [16]:
# ===================== CUT + CONCATENATE AUDIO =====================
import soundfile as sf
from scipy.signal import resample_poly

SR = 16000
utt_index = utts.set_index("utt_id")

def read_span(path, start, end):
    info = sf.info(path)
    x, sr = sf.read(path, start=int(round(start * info.samplerate)), stop=int(round(end * info.samplerate)),
                    dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if sr != SR:
        g = np.gcd(sr, SR); x = resample_poly(x, SR // g, sr // g).astype("float32")
    return x

gap = np.zeros(int(GAP_SEC * SR), dtype="float32")
audio_rows = []
for w in sel.itertuples():
    out = f"{WIN_DIR}/{w.window_id}.wav"
    ids = w.utt_ids.split()
    expected = w.speech_sec + GAP_SEC * (len(ids) - 1)
    if not os.path.exists(out):
        parts = []
        for i, u in enumerate(ids):
            r = utt_index.loc[u]
            parts.append(read_span(r.wav_path, r.start, r.end))
            if i < len(ids) - 1:
                parts.append(gap)
        sf.write(out, np.concatenate(parts), SR, subtype="PCM_16")
    dur = sf.info(out).duration
    assert abs(dur - expected) < 0.05 * len(ids) / 10 + 0.1, f"{w.window_id}: duration {dur:.2f}s vs {expected:.2f}s"
    with open(f"{WIN_DIR}/{w.window_id}.ref.txt", "w", encoding="utf-8") as f:
        f.write(w.ref_raw + "\n")
    audio_rows.append(dict(window_id=w.window_id, wav=out, audio_sec=round(dur, 3)))
sel = sel.merge(pd.DataFrame(audio_rows), on="window_id")
sel.to_csv(f"{OUT_DIR}/selected_windows.csv", index=False)
print("windows written:", len(audio_rows), "| total audio:", round(sel.audio_sec.sum() / 60, 1), "min")

windows written: 25 | total audio: 130.6 min


In [17]:
# ===================== MODEL PREPARATION =====================
from huggingface_hub import snapshot_download, model_info

def hf_sha(repo):
    try:
        return model_info(repo).sha
    except Exception as e:
        return f"unavailable ({type(e).__name__})"

def prepare_model(name):
    """Returns (path_or_size_for_faster_whisper, provenance dict)."""
    src = ASR_MODELS[name]["source"]
    if src.startswith("convert:"):
        repo = src.split(":", 1)[1]
        out = f"{MODELS_DIR}/{repo.replace('/', '__')}-ct2-fp16"
        if not os.path.exists(f"{out}/model.bin"):
            help_txt = subprocess.run(["ct2-transformers-converter", "--help"], capture_output=True, text=True).stdout
            cmd = ["ct2-transformers-converter", "--model", repo, "--output_dir", out, "--quantization", "float16",
                   "--copy_files", "preprocessor_config.json", "--force"]
            if "--low_cpu_mem_usage" in help_txt:
                cmd.append("--low_cpu_mem_usage")
            print("Converting (needs ~6 GB RAM, several minutes):", " ".join(cmd))
            subprocess.run(cmd, check=True)
        # faster-whisper silently falls back to the whisper-TINY tokenizer if tokenizer.json is missing,
        # which has different special-token ids from large-v3. The repo ships no tokenizer.json, so build it.
        if not os.path.exists(f"{out}/tokenizer.json"):
            from transformers import WhisperTokenizerFast
            WhisperTokenizerFast.from_pretrained(repo).backend_tokenizer.save(f"{out}/tokenizer.json")
        import tokenizers
        tok = tokenizers.Tokenizer.from_file(f"{out}/tokenizer.json")
        vocab_file = next((f"{out}/{v}" for v in ("vocabulary.json", "vocabulary.txt") if os.path.exists(f"{out}/{v}")), None)
        if vocab_file:
            n_ct2 = len(json.load(open(vocab_file))) if vocab_file.endswith(".json") else sum(1 for _ in open(vocab_file))
            n_tok = tok.get_vocab_size(with_added_tokens=True)
            assert n_ct2 == n_tok, f"vocab mismatch ct2={n_ct2} tokenizer={n_tok}"
        for t in ("<|hi|>", "<|en|>", "<|transcribe|>"):
            assert tok.token_to_id(t) is not None, f"missing special token {t}"
        pre = json.load(open(f"{out}/preprocessor_config.json"))
        print(f"{name}: feature_size={pre.get('feature_size')} | <|mixedcode|> id={tok.token_to_id('<|mixedcode|>')}")
        return out, dict(repo=repo, revision=hf_sha(repo), converted_to=out, feature_size=pre.get("feature_size"))
    if src.startswith("hfct2:"):
        _, repo, sub = src.split(":")
        p = snapshot_download(repo, allow_patterns=[f"{sub}/*"])
        return f"{p}/{sub}", dict(repo=repo, revision=hf_sha(repo), subdir=sub)
    return src, dict(repo=f"Systran/faster-whisper-{src}", revision=hf_sha(f"Systran/faster-whisper-{src}"))

def effective_decode(name):
    d = dict(ASR_MODELS[name]["decode"])
    if SEGMENTATION == "vad":
        d["without_timestamps"] = True     # each clip is <= MAX_CLIP_SEC; same setting for every model
    return d

def vad_clips(audio):
    """Silero VAD speech chunks merged greedily into clips of <= MAX_CLIP_SEC (cuts only at pauses).
    Returns [[start_sec, end_sec], ...]."""
    from faster_whisper.vad import get_speech_timestamps, VadOptions
    ts = get_speech_timestamps(audio, VadOptions(**VAD_PARAMS), sampling_rate=SR)
    clips = []
    for t in ts:
        s, e = t["start"] / SR, t["end"] / SR
        if clips and e - clips[-1][0] <= MAX_CLIP_SEC:
            clips[-1][1] = e
        else:
            clips.append([s, e])
    if not clips:                          # VAD found nothing: decode the whole file in fixed clips
        dur = len(audio) / SR
        clips = [[s, min(s + MAX_CLIP_SEC, dur)] for s in np.arange(0, dur, MAX_CLIP_SEC)]
    return clips

def get_transcriber(name):
    """Returns (fn(wav_path) -> (text, meta), provenance, cleanup_fn)."""
    from faster_whisper import WhisperModel
    path, prov = prepare_model(name)
    device = "cuda" if gpu else "cpu"
    ct = ASR_MODELS[name]["compute_type"] if device == "cuda" else "int8"
    model = WhisperModel(path, device=device, compute_type=ct)
    decode = effective_decode(name)
    def fn(wav):
        # Pass a float32 16 kHz array, not a path: faster-whisper 1.2.1 decodes paths with PyAV using
        # av.open(..., metadata_errors=...), an argument newer PyAV releases removed (TypeError on Colab).
        audio, sr = sf.read(wav, dtype="float32", always_2d=True)
        audio = audio.mean(axis=1)
        assert sr == SR, f"{wav}: expected {SR} Hz, got {sr}"
        d = dict(decode)
        clips = []
        if SEGMENTATION == "vad":
            clips = vad_clips(audio)
            d["clip_timestamps"] = [t for c in clips for t in c]
        segs, info = model.transcribe(audio, **d)
        segs = list(segs)                                     # generator: decoding happens here
        text = " ".join(s.text.strip() for s in segs).strip()
        ntok = [len(s.tokens) for s in segs]
        meta = dict(n_asr_segments=len(segs), detected_lang=info.language,
                    mean_avg_logprob=float(np.mean([s.avg_logprob for s in segs])) if segs else np.nan,
                    max_compression_ratio=float(max([s.compression_ratio for s in segs], default=np.nan)),
                    max_segment_tokens=max(ntok, default=0),
                    n_segments_at_token_limit=sum(n >= 440 for n in ntok),   # Whisper max_length = 448 incl. prompt
                    n_clips=len(clips),
                    max_clip_sec=round(max((e - s for s, e in clips), default=np.nan), 2),
                    vad_coverage=round(sum(e - s for s, e in clips) / (len(audio) / SR), 4) if clips else np.nan)
        return text, meta
    def cleanup():
        nonlocal model
        del model; gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception:
            pass
    prov.update(device=device, compute_type=ct)
    return fn, prov, cleanup

In [18]:
# ===================== RUN ASR (resumable) =====================
HYP_CSV = f"{OUT_DIR}/asr_hyps_{ASR_TAG}.csv"
print("ASR output tag:", ASR_TAG, "->", HYP_CSV)
done = set()
if os.path.exists(HYP_CSV):
    prev = pd.read_csv(HYP_CSV, keep_default_na=False)
    done = set(zip(prev.window_id, prev.model))

provenance = {}
for name in RUN_MODELS:
    todo = [w for w in sel.itertuples() if (w.window_id, name) not in done]
    if not todo:
        print(f"{name}: all windows done"); continue
    fn, prov, cleanup = get_transcriber(name)
    provenance[name] = prov
    log_prompt(step="1-asr", component="asr_decoding", model=name,
               prompt=ASR_MODELS[name]["decode"].get("initial_prompt"),
               params=dict(decode=effective_decode(name), provenance=prov, segmentation=SEGMENTATION,
                           asr_tag=ASR_TAG, max_clip_sec=MAX_CLIP_SEC,
                           vad_params=VAD_PARAMS if SEGMENTATION == "vad" else None,
                           note="no text prompt; forced language token; deterministic beam search"))
    for w in todo:
        t0 = time.time()
        text, meta = fn(w.wav)
        dt = time.time() - t0
        row = dict(window_id=w.window_id, model=name, hyp_raw=text, runtime_sec=round(dt, 2),
                   rtf=round(dt / w.audio_sec, 4), **meta)
        pd.DataFrame([row]).to_csv(HYP_CSV, mode="a", header=not os.path.exists(HYP_CSV), index=False)
        print(f"{name} {w.window_id}: {dt:.1f}s (RTF {dt / w.audio_sec:.3f}) | {text[:80]}")
    cleanup()
update_run_config(asr_provenance=provenance)

ASR output tag: vad15 -> /content/drive/MyDrive/hinglish_asr_sum/outputs/asr_hyps_vad15.csv
whisper-small: all windows done
Converting (needs ~6 GB RAM, several minutes): ct2-transformers-converter --model Trelis/whisper-hinglish-preview --output_dir /content/mucs_data/models/Trelis__whisper-hinglish-preview-ct2-fp16 --quantization float16 --copy_files preprocessor_config.json --force --low_cpu_mem_usage
trelis-hinglish-preview: feature_size=128 | <|mixedcode|> id=51866
trelis-hinglish-preview W21: 56.7s (RTF 0.180) | लिबर ऑफिस इम्प्रेस में प्रस्तुति को प्रिंट करने पर इस स्पोकेन टूटोरियल में आपका 
trelis-hinglish-preview W22: 50.5s (RTF 0.161) | नमस्कार फीचर्स ऑफ जे कैंपेंट के इस टोटोरियल में आपका स्वागत है इस टोटोरियल में ह
trelis-hinglish-preview W23: 56.4s (RTF 0.180) | लिबर ऑफिस इम प्रेस में प्रस्तुति देखने पर टीटूरियल में आपका स्वागत है इस टीटूरिय
trelis-hinglish-preview W24: 39.7s (RTF 0.129) | एक्सपी के प्रयोग से फीडबैक डायग्राम बनाने के इस मौखिक टुटोरियल में आपका स्वागत ह


In [19]:
# ===================== METRICS PER WINDOW =====================
import jiwer

def align_stats(ref, hyp, ref_scripts):
    """Word-level S/D/I/H plus per-language error counts over REFERENCE tokens (insertions are unattributed)."""
    n = len(ref.split())
    tot = Counter(ref_scripts)
    if not hyp.strip():
        return dict(S=0, D=n, I=0, H=0, N=n, **{f"ref_{k}": tot[k] for k in ("hi", "en")},
                    **{f"err_{k}": tot[k] for k in ("hi", "en")})
    o = jiwer.process_words(ref, hyp)
    err = Counter()
    for ch in o.alignments[0]:
        if ch.type != "equal":
            for i in range(ch.ref_start_idx, ch.ref_end_idx):
                err[ref_scripts[i]] += 1
    return dict(S=o.substitutions, D=o.deletions, I=o.insertions, H=o.hits, N=n,
                **{f"ref_{k}": tot[k] for k in ("hi", "en")}, **{f"err_{k}": err[k] for k in ("hi", "en")})

def char_stats(ref, hyp):
    if not hyp:
        return dict(cS=0, cD=len(ref), cI=0, cN=len(ref))
    o = jiwer.process_characters(ref, hyp)
    return dict(cS=o.substitutions, cD=o.deletions, cI=o.insertions, cN=len(ref))

def max_ngram_repeats(s, n=6):
    w = s.split()
    c = Counter(tuple(w[i:i + n]) for i in range(len(w) - n + 1))
    return max(c.values(), default=0)

def zlib_ratio(s):
    b = s.encode("utf-8")
    return len(b) / max(1, len(zlib.compress(b)))

hyps = pd.read_csv(HYP_CSV, keep_default_na=False)
hyps = hyps[hyps.model.isin(RUN_MODELS)].drop_duplicates(["window_id", "model"], keep="last")
refs = sel.set_index("window_id")
rows = []
for h in hyps.itertuples():
    ref = refs.loc[h.window_id, "ref_norm"]
    hyp = norm_text(h.hyp_raw)
    scripts = [token_script(t) for t in ref.split()]
    ref_r, hyp_r = romanize_text(ref), romanize_text(hyp)
    assert len(ref_r.split()) == len(scripts)
    raw = align_stats(ref, hyp, scripts)
    tr = align_stats(ref_r, hyp_r, scripts)
    cr, ctr = char_stats(ref, hyp), char_stats(ref_r, hyp_r)
    hs = script_stats(hyp)
    rows.append(dict(
        window_id=h.window_id, model=h.model, n_ref_words=raw["N"], n_hyp_words=len(hyp.split()),
        wer=(raw["S"] + raw["D"] + raw["I"]) / raw["N"],
        cer=(cr["cS"] + cr["cD"] + cr["cI"]) / cr["cN"],
        translit_wer=(tr["S"] + tr["D"] + tr["I"]) / tr["N"],
        translit_cer=(ctr["cS"] + ctr["cD"] + ctr["cI"]) / ctr["cN"],
        sub=raw["S"], dele=raw["D"], ins=raw["I"],
        tr_sub=tr["S"], tr_dele=tr["D"], tr_ins=tr["I"],
        ref_hi_tokens=raw["ref_hi"], ref_en_tokens=raw["ref_en"],
        hi_token_err=raw["err_hi"] / raw["ref_hi"] if raw["ref_hi"] else np.nan,
        en_token_err=raw["err_en"] / raw["ref_en"] if raw["ref_en"] else np.nan,
        tr_hi_token_err=tr["err_hi"] / tr["ref_hi"] if tr["ref_hi"] else np.nan,
        tr_en_token_err=tr["err_en"] / tr["ref_en"] if tr["ref_en"] else np.nan,
        hyp_frac_hi=hs["frac_hi"], hyp_frac_en=hs["frac_en"],
        len_ratio=len(hyp.split()) / raw["N"], hyp_zlib_ratio=zlib_ratio(hyp),
        rel_zlib_ratio=zlib_ratio(hyp) / zlib_ratio(ref),
        max_6gram_repeats=max_ngram_repeats(hyp), ref_max_6gram_repeats=max_ngram_repeats(ref),
        asr_max_compression_ratio=h.max_compression_ratio,
        # raw counts kept for exact corpus-level aggregation
        _err=raw["S"] + raw["D"] + raw["I"], _N=raw["N"], _cerr=cr["cS"] + cr["cD"] + cr["cI"], _cN=cr["cN"],
        _trerr=tr["S"] + tr["D"] + tr["I"], _trcerr=ctr["cS"] + ctr["cD"] + ctr["cI"], _trcN=ctr["cN"],
        _hi_err=raw["err_hi"], _hi_N=raw["ref_hi"], _en_err=raw["err_en"], _en_N=raw["ref_en"],
        _trhi_err=tr["err_hi"], _tren_err=tr["err_en"],
        hyp_norm=hyp))
per = pd.DataFrame(rows).sort_values(["model", "window_id"])
per["flag_length"] = (per.len_ratio > 1.3) | (per.len_ratio < 0.7)
# Repetition loop = some 6-word sequence occurs >= 10 times in the hypothesis, or the hypothesis is much more
# compressible than its own reference. (v1 also used Whisper's per-chunk compression ratio > 2.4; on the real run
# that fired on 23/25 fine-tune windows whose hypotheses contained no 6-gram more than 5 times, so it was dropped.
# asr_max_compression_ratio is still saved for reporting.)
per["flag_repetition"] = (per.max_6gram_repeats >= 10) | (per.rel_zlib_ratio > 1.3)
per.to_csv(f"{OUT_DIR}/asr_metrics_per_window_{ASR_TAG}.csv", index=False)
print(per[["window_id", "model", "wer", "cer", "translit_wer", "hi_token_err", "en_token_err",
           "len_ratio", "flag_length", "flag_repetition"]].round(3).to_string(index=False))

window_id                   model   wer   cer  translit_wer  hi_token_err  en_token_err  len_ratio  flag_length  flag_repetition
      W00 trelis-hinglish-preview 0.263 0.232         0.254         0.197         0.535      1.023        False            False
      W01 trelis-hinglish-preview 0.197 0.136         0.176         0.122         0.291      1.014        False            False
      W02 trelis-hinglish-preview 0.328 0.300         0.288         0.096         0.726      1.017        False            False
      W03 trelis-hinglish-preview 0.349 0.317         0.316         0.143         0.673      1.000        False            False
      W04 trelis-hinglish-preview 0.174 0.125         0.166         0.080         0.296      1.013        False            False
      W05 trelis-hinglish-preview 0.345 0.284         0.312         0.255         0.574      1.042        False            False
      W06 trelis-hinglish-preview 0.357 0.351         0.327         0.078         0.422      1.13

In [20]:
# ===================== CORPUS-LEVEL METRICS, BOOTSTRAP CIs, PAIRED TEST =====================
from scipy.stats import wilcoxon

METRICS = {  # name: (error-count column, denominator column)
    "wer": ("_err", "_N"), "cer": ("_cerr", "_cN"), "translit_wer": ("_trerr", "_N"),
    "translit_cer": ("_trcerr", "_trcN"), "hi_token_err": ("_hi_err", "_hi_N"), "en_token_err": ("_en_err", "_en_N"),
    "translit_hi_token_err": ("_trhi_err", "_hi_N"), "translit_en_token_err": ("_tren_err", "_en_N")}

win_ids = sorted(sel.window_id)
n_w = len(win_ids)
boot_idx = np.random.default_rng(SEED).integers(0, n_w, size=(N_BOOT, n_w))   # shared across models -> paired

def mat(model, col):
    d = per[per.model == model].set_index("window_id")
    return d.reindex(win_ids)[col].to_numpy(dtype=float)

summary = []
for m in RUN_MODELS:
    if m not in set(per.model):
        continue
    for name, (e, n) in METRICS.items():
        E, N = mat(m, e), mat(m, n)
        point = E.sum() / N.sum()
        bs = E[boot_idx].sum(1) / N[boot_idx].sum(1)
        summary.append(dict(model=m, metric=name, corpus_value=point,
                            ci95_lo=np.percentile(bs, 2.5), ci95_hi=np.percentile(bs, 97.5),
                            mean_per_window=np.nanmean(E / np.where(N > 0, N, np.nan)), n_windows=n_w))
summ = pd.DataFrame(summary)
summ.to_csv(f"{OUT_DIR}/asr_metrics_summary_{ASR_TAG}.csv", index=False)
print(summ.round(4).to_string(index=False))

paired = []
if len([m for m in RUN_MODELS if m in set(per.model)]) == 2:
    a, b = RUN_MODELS
    for name, (e, n) in METRICS.items():
        Ea, Na, Eb, Nb = mat(a, e), mat(a, n), mat(b, e), mat(b, n)
        diff = Ea.sum() / Na.sum() - Eb.sum() / Nb.sum()
        bs = Ea[boot_idx].sum(1) / Na[boot_idx].sum(1) - Eb[boot_idx].sum(1) / Nb[boot_idx].sum(1)
        ra, rb = Ea / np.where(Na > 0, Na, np.nan), Eb / np.where(Nb > 0, Nb, np.nan)
        ok = ~np.isnan(ra) & ~np.isnan(rb)
        try:
            p = wilcoxon(ra[ok], rb[ok]).pvalue
        except ValueError:
            p = np.nan                     # e.g. all differences zero
        paired.append(dict(metric=name, model_a=a, model_b=b, diff_a_minus_b=diff,
                           ci95_lo=np.percentile(bs, 2.5), ci95_hi=np.percentile(bs, 97.5),
                           wilcoxon_p=p, n_windows=int(ok.sum()),
                           windows_a_worse=int((ra[ok] > rb[ok]).sum())))
    pd.DataFrame(paired).to_csv(f"{OUT_DIR}/asr_paired_tests_{ASR_TAG}.csv", index=False)
    print(pd.DataFrame(paired).round(4).to_string(index=False))
update_run_config(finished=now_utc())

                  model                metric  corpus_value  ci95_lo  ci95_hi  mean_per_window  n_windows
          whisper-small                   wer        0.6971   0.6721   0.7239           0.7004         25
          whisper-small                   cer        0.5456   0.5170   0.5756           0.5487         25
          whisper-small          translit_wer        0.6589   0.6322   0.6864           0.6618         25
          whisper-small          translit_cer        0.3769   0.3528   0.4023           0.3792         25
          whisper-small          hi_token_err        0.4908   0.4563   0.5272           0.4845         25
          whisper-small          en_token_err        0.9645   0.9456   0.9804           0.9579         25
          whisper-small translit_hi_token_err        0.4472   0.4111   0.4853           0.4401         25
          whisper-small translit_en_token_err        0.9370   0.9156   0.9546           0.9295         25
trelis-hinglish-preview                   wer 

In [21]:
# ===================== QUICK DIAGNOSTICS FOR THE CHECKLIST =====================
print("Flagged windows (length/repetition):")
print(per[per.flag_length | per.flag_repetition][["window_id", "model", "len_ratio", "rel_zlib_ratio",
                                                  "max_6gram_repeats", "asr_max_compression_ratio"]]
      .to_string(index=False) or "  none")
print("\nOutput script mix (mean fraction of hyp tokens):")
print(per.groupby("model")[["hyp_frac_hi", "hyp_frac_en"]].mean().round(3))
print("Reference script mix:", sel[["frac_hi", "frac_en"]].mean().round(3).to_dict())

print("\nSegmentation diagnostics (", ASR_TAG, "): deletions/insertions per model and chunking stats")
_h = pd.read_csv(HYP_CSV, keep_default_na=False)
_cols = [c for c in ["n_asr_segments", "n_clips", "max_clip_sec", "vad_coverage", "max_segment_tokens",
                     "n_segments_at_token_limit"] if c in _h.columns]
for c in _cols:
    _h[c] = pd.to_numeric(_h[c], errors="coerce")
print(per.groupby("model")[["sub", "dele", "ins"]].sum())
if _cols:
    print(_h.groupby("model")[_cols].agg(["mean", "min", "max"]).round(3).T.to_string())
low_cov = _h[_h.vad_coverage < 0.9] if "vad_coverage" in _h.columns else _h.iloc[0:0]
if len(low_cov):
    print("WARNING: VAD covered < 90% of the audio in:", sorted(set(low_cov.window_id)))

ex = sel.window_id.iloc[0]
print(f"\n=== Side-by-side, first 60 words of {ex} ===")
print("REF :", " ".join(sel.set_index("window_id").loc[ex, "ref_norm"].split()[:60]))
for m in RUN_MODELS:
    r = per[(per.model == m) & (per.window_id == ex)]
    if len(r):
        print(f"{m}:", " ".join(r.hyp_norm.iloc[0].split()[:60]))
print("\nFiles:", sorted(os.listdir(OUT_DIR)), sorted(os.listdir(LOG_DIR)))

Flagged windows (length/repetition):
window_id         model  len_ratio  rel_zlib_ratio  max_6gram_repeats  asr_max_compression_ratio
      W02 whisper-small   0.838608        1.801402                 19                  14.516129
      W03 whisper-small   0.872910        1.444031                 20                  13.636364
      W04 whisper-small   0.978862        1.307670                 17                  10.555556
      W05 whisper-small   0.911003        1.117910                 15                   8.351852
      W07 whisper-small   1.052189        1.104625                 19                  13.104167
      W09 whisper-small   0.860027        1.337432                 21                  14.302326
      W10 whisper-small   1.049774        1.358153                 31                  12.763158
      W14 whisper-small   0.800648        1.088568                 12                  10.318182
      W15 whisper-small   1.017857        1.396401                 51                  19.

In [25]:
# --- STEP 1 v2 SANITY CHECK (read-only) ---
_h = pd.read_csv(HYP_CSV, keep_default_na=False)
_p = pd.read_csv(f"{OUT_DIR}/asr_metrics_per_window_{ASR_TAG}.csv", keep_default_na=False)
for c in ["max_segment_tokens", "vad_coverage"]:
    _h[c] = pd.to_numeric(_h[c], errors="coerce")
for c in ["dele", "n_ref_words", "wer"]:
    _p[c] = pd.to_numeric(_p[c], errors="coerce")
_p["del_rate"] = _p.dele / _p.n_ref_words
t = _h.merge(_p[["window_id", "model", "wer", "del_rate"]], on=["window_id", "model"])
t = t.pivot(index="window_id", columns="model", values=["max_segment_tokens", "vad_coverage", "del_rate", "wer"])
print(t.round(3).to_string())

tr = _h[_h.model == "trelis-hinglish-preview"]
print("\nTrelis windows with a segment >= 215 tokens (possible truncation):",
      sorted(tr[tr.max_segment_tokens >= 215].window_id))
d = _p[_p.model == "trelis-hinglish-preview"].set_index("window_id").del_rate
print("Trelis deletion rate: median", round(d.median(), 3), "| W07", round(d["W07"], 3), "| W24", round(d["W24"], 3))
s = pd.read_csv(f"{OUT_DIR}/asr_metrics_summary_{ASR_TAG}.csv")
print(s[s.metric.isin(["wer", "translit_wer", "hi_token_err", "en_token_err"])].round(4).to_string(index=False))

               max_segment_tokens                          vad_coverage                              del_rate                                   wer              
model     trelis-hinglish-preview whisper-small trelis-hinglish-preview whisper-small trelis-hinglish-preview whisper-small trelis-hinglish-preview whisper-small
window_id                                                                                                                                                        
W00                         193.0         224.0                   0.932         0.932                   0.005         0.141                   0.263         0.596
W01                         182.0         224.0                   0.950         0.950                   0.012         0.105                   0.197         0.655
W02                         204.0         224.0                   0.964         0.964                   0.021         0.212                   0.328         0.790
W03                         

In [22]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
import pandas as pd

folder = "/content/drive/MyDrive/hinglish_asr_sum/outputs"

print("Files found:")
for f in glob.glob(folder + "/*"):
    print(os.path.basename(f), round(os.path.getsize(f)/1024, 1), "KB")

csv = folder + "/asr_hyps_vad15.csv"

if os.path.exists(csv):
    df = pd.read_csv(csv)
    print("\nCompleted windows:")
    print(df.groupby("model")["window_id"].agg(["count", "unique"]))
else:
    print("\nASR CSV not found yet.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files found:
asr_hyps.csv 314.2 KB
asr_metrics_summary.csv 1.8 KB
asr_metrics_per_window.csv 326.8 KB
asr_paired_tests.csv 1.2 KB
asr_hyps_vad15.csv 372.0 KB
duplicate_recordings.csv 0.0 KB
candidate_windows.csv 682.9 KB
selected_windows.csv 362.3 KB
selected_windows_before_resume.csv 362.3 KB
asr_hyps_vad15_backup_46rows.csv 345.0 KB
asr_metrics_per_window_vad15.csv 384.7 KB
asr_metrics_summary_vad15.csv 1.8 KB
asr_paired_tests_vad15.csv 1.2 KB

Completed windows:
                         count  \
model                            
trelis-hinglish-preview     25   
whisper-small               25   

                                                                    unique  
model                                                                       
trelis-hinglish-preview  [W00, W01, W02, W03, W04, W05, W06, W07, W08, ...  
whisper-small            [W00, W0

In [24]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/hinglish_asr_sum/outputs/asr_hyps_vad15.csv"
)

all_windows = ["W%02d" % i for i in range(25)]

done = df[df["model"] == "trelis-hinglish-preview"]["window_id"].unique()

missing = [w for w in all_windows if w not in done]

print("Missing windows:", missing)

Missing windows: []


## Before Step 2 — check these (in this order)

1. **Kaldi parse mode** printed `mode=kaldi_segments`. If it says `fallback`, open a few utterance ids and confirm
   the recording grouping before trusting any window.
2. **Duplicates:** open `duplicate_recordings.csv`. Near-duplicate tutorials inflate agreement and should not both
   be in the sample (the code keeps one per group — confirm that grouping looks right).
3. **Listen to 2–3 windows** (`/content/mucs_data/windows/W00.wav`) while reading `W00.ref.txt`. Joins must be
   clean and the transcript must match the audio from start to end.
4. **Reference conventions:** MUCS references are not consistent (the same English word sometimes appears in
   Devanagari). Look at how big the gap between `wer` and `translit_wer` is — that gap is script-convention noise,
   not recognition error, and you must say which one you report as primary.
5. **Hallucination flags:** any window with `flag_length` or `flag_repetition` — read the hypothesis. Decide
   *before* Step 2 whether to keep it (it is a real ASR failure) and state the rule in the paper.
6. **Fine-tune output script:** `hyp_frac_en` for the fine-tune should be close to the reference `frac_en`. If the
   fine-tune writes English words in Devanagari (or everything in Roman), raw WER is meaningless for it.
7. **Contamination:** the Trelis card does not list training data; `abhishekgautamm` trained on MUCS train;
   `Deepro713/whisper-large-v3-turbo-hinglish` trained on `agarwalayushi/hinglish`, whose rows are Spoken Tutorial
   audio (likely MUCS-derived, unverified). Write this as a limitation; if the fine-tune's WER looks implausibly low,
   suspect test-set overlap.
8. **Per-language error rates** (`hi_token_err` vs `en_token_err`): these are the realistic error levels for
   Step 2's noise-injection sweeps — anchor the sweep grid so it covers them.
9. **Segmentation:** in the diagnostics, `vad_coverage` should be close to the share of the window that is speech
   (above 0.9), `n_segments_at_token_limit` should be 0, and the fine-tune's deletions should drop compared with
   the `fixed30` run. Keep the `fixed30` files: they are the before/after evidence for the paper's method section.
10. **Logs:** `logs/prompts_log.jsonl` has one entry per ASR model; `logs/run_config_step1.json` has package
   versions, GPU, tarball sha256 and model revisions. Copy both into the appendix.